In [1]:
from stock_embeddings import ReturnsData
from stock_embeddings.context.window import get_target_context_sets, Euclidean, ICCBRMetric

# Constants and configurations
TRAIN_PCT = 1
CONTEXT_SIZE = 32
GRANULARITY = 1 # 1=daily, 5=weekly, etc.
WINDOW_LENGTHS = [20]
STRIDE = None

data = ReturnsData(
    daily_returns_path="../Data/returns_df_611.csv",
    extras_path="../Data/historical_stocks.csv",
)

data = ReturnsData(
        daily_returns_path="../Data/returns_df_611.csv",
        extras_path="../Data/historical_stocks.csv",
    )
data.change_returns_period(GRANULARITY)
data.train_test_split(TRAIN_PCT)
    
tgt_context_sets = []
for window_length in WINDOW_LENGTHS:
    print(f"Running for period {window_length}")
    
    tgt_context_sets += get_target_context_sets(
        X=data.returns_df.select(data.tickers).to_numpy().T,
        metric_class=Euclidean(),
        window_length=window_length,
        stride=STRIDE,
        context_size=32,
        verbose=True,
    )
    print("="*20)
print(f"Total number of target context sets: {len(tgt_context_sets)}")

No change made because period entered is 1
Running for period 20


  0%|          | 0/234 [00:00<?, ?it/s]

 10%|▉         | 23/234 [00:00<00:00, 226.46it/s]

 20%|█▉        | 46/234 [00:00<00:01, 179.60it/s]

 29%|██▉       | 69/234 [00:00<00:00, 196.63it/s]

...

 79%|███████▊  | 184/234 [00:00<00:00, 214.64it/s]

 89%|████████▉ | 208/234 [00:01<00:00, 220.04it/s]

 99%|█████████▊| 231/234 [00:01<00:00, 191.03it/s]

100%|██████████| 234/234 [00:01<00:00, 199.68it/s]

Total number of target context sets: 142974


In [2]:
import polars as pl
import numpy as np

ticker = "JPM"
i = data.ticker2idx[ticker]
temp = np.array([xi[1] for xi in tgt_context_sets if xi[0] == i]).flatten()
top_values = pl.Series("value", temp).value_counts(sort=True)["value"].to_list()
print(
    f"The most commonly co-occurring stocks with {ticker} are: {[data.idx2ticker[xi] for xi in top_values][:5]}"
)

The most commonly co-occurring stocks with JPM are: ['C', 'WFC', 'PNC', 'USB', 'BAC']


In [3]:
from stock_embeddings import ClassificationEmbeddings

EMBEDDING_DIM = 20
model = ClassificationEmbeddings(
    n_time_series=len(data.tickers), embedding_dim=EMBEDDING_DIM
)

In [4]:
from stock_embeddings import train_embeddings_from_tgt_context_sets


EPOCHS = 10
model, losses = train_embeddings_from_tgt_context_sets(
    n_time_series=len(data.tickers),
    tgt_context_sets=tgt_context_sets,
    model=model,
    epochs=EPOCHS,
    batch_size=64,
    early_stopping=False,
    device="cpu",
    # embedding_dim=EMBEDDING_DIM,
    verbose=True,
)

Training embeddings...


  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:01<00:11,  1.33s/it]

Epoch 0: Loss = 0.10526631683431502


 20%|██        | 2/10 [00:02<00:10,  1.32s/it]

...

 90%|█████████ | 9/10 [00:11<00:01,  1.31s/it]

Epoch 8: Loss = 0.09984448670580202


100%|██████████| 10/10 [00:13<00:00,  1.31s/it]

100%|██████████| 10/10 [00:13<00:00,  1.31s/it]

Epoch 9: Loss = 0.09905794523806515


In [5]:
from stock_embeddings import get_sector_score

get_sector_score(model.embeddings.weight.detach().numpy(), sectors=data.sectors)

/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

Precision Score: 0.27
Recall Score: 0.29
F1 Score: 0.27
Accuracy Score: 0.29


/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

In [6]:
SAVE_MODEL = False
SAVE_PATH_TEMPLATE = "embeddings/abs_diff_E{epochs}_C{context_size}_D{embedding_dim}_P{periods}_train{train_pct}.pt"